In [ ]:
from torchvision.datasets import ImageFolder
from pathlib import Path


CROPS_0PCT_SPLIT_DIR = Path("../data/Iron-Scraps/classification_split/crops_0pct_split")
CROPS_10PCT_SPLIT_DIR = Path("../data/Iron-Scraps/classification_split/crops_10pct_split")
CROPS_25PCT_SPLIT_DIR = Path("../data/Iron-Scraps/classification_split/crops_25pct_split")

In [3]:
from torchvision import transforms
from torchvision.transforms import functional as F


class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = max(w, h)

        pad_left = int((max_wh - w) / 2)
        pad_top = int((max_wh - h) / 2)
        pad_right = max_wh - w - pad_left
        pad_bottom = max_wh - h - pad_top

        padding = (pad_left, pad_top, pad_right, pad_bottom)

        return F.pad(img=image, padding=padding)


transform = transforms.Compose([
    SquarePad(),
    transforms.Resize(size=[224, ]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# 파이토치의 이미지폴더는 경로 폴더 하위 폴더를 바로 클래스명으로 인식합니다.
# 따라서, 경로를 정확하게 클래스 폴더 상위폴더로 잡아줘야 함.
# 이미지폴더로 생성한 객체는 데이터셋 타입으로 데이터로더에 즉시 인자로 넣을 수 있다.

train_dataset = ImageFolder(
    root=CROPS_25PCT_SPLIT_DIR / "train",
    transform=transform)
test_dataset = ImageFolder(
    root=CROPS_25PCT_SPLIT_DIR / "test",
    transform=transform)

print(train_dataset)
print(test_dataset)

Dataset ImageFolder
    Number of datapoints: 14534
    Root location: ../data/Iron-Scraps/classification_split/crops_25pct_split/train
    StandardTransform
Transform: Compose(
               Resize(size=[224], interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )
Dataset ImageFolder
    Number of datapoints: 3369
    Root location: ../data/Iron-Scraps/classification_split/crops_25pct_split/test
    StandardTransform
Transform: Compose(
               Resize(size=[224], interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )


In [5]:
from torch.utils.data import Dataset, DataLoader
from typing import Tuple


BATCH_SIZE = 32


def get_data_loader() -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
    shuffled_test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
    )

    return train_loader, test_loader, shuffled_test_loader

In [8]:
import torch


dinov3_vits16 = torch.hub.load("../dinov3", 'dinov3_vits16', source='local', weights="../models/dino/weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth")

In [9]:
dinov3_vits16

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    )
  )
  (norm): LayerN

In [ ]:
from torch import nn
from peft import LoraConfig, get_peft_model


NUM_CLASSES = 3
LORA_RANK = 16
LORA_ALPHA = 16
TARGET_MODULES = ["qkv"]


def set_model(r: int, lora_alpha: int, target_modules: list) -> nn.Module:
    dinov3_vits16.head = nn.Sequential(
        nn.Linear(384, 128),
        nn.LayerNorm(128),
        nn.GELU(),
        nn.Dropout(),
        nn.Linear(128, NUM_CLASSES),
    )

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=0.5,
        target_modules=target_modules,
        bias="none",
        modules_to_save=["head"],
    )

    peft_model = get_peft_model(
        
        model=dinov3_vits16,
        peft_config=lora_config,
    )

    return peft_model

In [ ]:
from torch import optim
from tqdm import tqdm

 
NUM_EPOCHS = 50


optimizer = optim.AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=NUM_EPOCHS)


def train_model(device, train_loader, test_loader, model, criterion, optimizer):
    model.to(device)
    model.train()
    
    running_loss = 0.0
    progress_bar = tqdm(iterable=train_loader, desc="Training")
    for index, batch in enumerate(progress_bar):
        optimizer.zero_grad()
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)
        
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            logits = model(images)
            loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        avg_loss = running_loss / (index + 1)

        current_lr = scheduler.get_last_lr()[0]
        progress_bar.set_postfix({
            "avg_loss": f"{avg_loss:.4f}",
            "current_lr": f"{current_lr:.6f}",
        })